In [3]:
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal

# Define output directory for spectrogram files and create it if not exists
output_dir = 'spectrogram_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def load_audio_and_annotation(file_base_name):
    # Load the WAV audio file
    audio_path = file_base_name + '.wav'
    sample_rate, samples = wavfile.read(audio_path)
    
    # Load the annotation file (typically .txt or .csv)
    annot_file_path = file_base_name + '.Table.1.selections.txt'
    df = pd.read_csv(annot_file_path, sep='\t')
    
    return sample_rate, samples, df

def extract_audio_segments(samples, sample_rate, annotations):
    segments = []
    for _, row in annotations.iterrows():
        start_time, end_time = row['Begin Time (s)'], row['End Time (s)']
        start_sample, end_sample = int(start_time * sample_rate), int(end_time * sample_rate)
        
        # Check if start and end indices are valid
        if start_sample >= end_sample or start_sample < 0 or end_sample > len(samples):
            print(f"Invalid segment: {start_time}-{end_time}s")
            continue
        
        segment = samples[start_sample:end_sample]
        segments.append({
            'Selection': row['Selection'],
            'Annotation': row['Annotation'],
            'Begin Time (s)': start_time,
            'End Time (s)': end_time,
            'audio_segment': segment,
            'Low Freq (Hz)': row['Low Freq (Hz)'],
            'High Freq (Hz)': row['High Freq (Hz)']
        })
    return segments

def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=1024, nfft=2048, noverlap=512):  
    frequencies, times, spectrogram_data = signal.spectrogram(
        audio_segment, sample_rate, nperseg=nperseg, nfft=nfft, noverlap=noverlap, window='hann'
    )
    spectrogram_data = np.maximum(spectrogram_data, 1e-10)
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    return frequencies[freq_slice], times, spectrogram_data[freq_slice]

def pad_or_crop_spectrogram(spectrogram_data, target_shape):
    # Ensure spectrogram data is a numpy array
    spectrogram_data = np.array(spectrogram_data)

    # Check if the spectrogram is 2D (frequency x time)
    if spectrogram_data.ndim == 2:
        current_shape = spectrogram_data.shape
        padded_spectrogram = np.zeros(target_shape)

        # Check if the current spectrogram size is within bounds
        for i in range(min(current_shape[0], target_shape[0])):  # Frequencies
            for j in range(min(current_shape[1], target_shape[1])):  # Time steps
                padded_spectrogram[i, j] = float(spectrogram_data[i, j])
        return padded_spectrogram
    else:
        print(f"Error: Spectrogram data has invalid dimensions {spectrogram_data.shape}. Expected 2D.")
        raise ValueError(f"Spectrogram data must be 2D with dimensions (frequency, time), but got shape {spectrogram_data.shape}.")

def create_no_call_segments(samples, sample_rate, call_annotations, silence_duration_threshold=0.5):
    # Get the end times of existing calls
    call_end_times = call_annotations['End Time (s)'].values
    call_start_times = call_annotations['Begin Time (s)'].values
    
    # Sort calls by start times (just in case)
    sorted_start_times = np.sort(call_start_times)
    sorted_end_times = call_end_times[np.argsort(call_start_times)]
    
    # Generate the 'no-call' periods by looking at gaps between successive call end times and next start times
    no_call_segments = []
    for i in range(len(sorted_start_times) - 1):
        end_of_previous_call = sorted_end_times[i]
        start_of_next_call = sorted_start_times[i + 1]
        
        # Check for gaps larger than the silence threshold
        if start_of_next_call - end_of_previous_call > silence_duration_threshold:
            # Append the no-call segment between two calls
            start_sample = int(end_of_previous_call * sample_rate)
            end_sample = int(start_of_next_call * sample_rate)
            no_call_segment = samples[start_sample:end_sample]
            no_call_segments.append(no_call_segment)
    
    # Optionally, add the silent period after the last call
    last_call_end_time = sorted_end_times[-1]
    audio_duration = len(samples) / sample_rate
    if audio_duration - last_call_end_time > silence_duration_threshold:
        start_sample = int(last_call_end_time * sample_rate)
        no_call_segment = samples[start_sample:]
        no_call_segments.append(no_call_segment)

    return no_call_segments

def get_category_name(annotation):
    category_mapping = {
        'Rupe A': 'rupe_A',
        'Rupe B': 'rupe_B',
        'Rupe C': 'rupe_C',
        'Growl B': 'growl_B',
        'Moan': 'moan',
        'G rupe': 'g_rupe',
        'Guttural rupe': 'g_rupe',
        'no_call': 'no_call',  
        'Type 4 A': 'type_4_A'
    }
    # Logging the unknown category
    if annotation not in category_mapping:
        print(f"Unknown annotation found: {annotation}")
    return category_mapping.get(annotation, 'unknown')

def calculate_baselines(folder_paths):
    """
    Calculate the baseline segment duration and frequency range across all files in the folder paths.
    """
    global max_time, max_frequency
    
    max_time, max_frequency = 0, 0
    
    # Iterate through each folder in the folder_paths list
    for folder_path in folder_paths:
        # Loop through each file in the current folder
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_base_name = os.path.splitext(file)[0]
                file_base_path = os.path.join(folder_path, file_base_name)
                
                # Load the audio and annotation
                sample_rate, samples, df = load_audio_and_annotation(file_base_path)
                
                # Extract segments for the current file
                segments = extract_audio_segments(samples, sample_rate, df)
                
                # Find the longest duration in terms of time
                for segment in segments:
                    max_time = max(max_time, segment['End Time (s)'] - segment['Begin Time (s)'])
                    max_frequency = max(max_frequency, segment['High Freq (Hz)'] - segment['Low Freq (Hz)'])

# Folder paths for the WAV and annotation files
folder_paths = [
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Guttural rupe',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Rupes A and B',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Moan'
]

dataset = []

# Calculate the baseline values for time and frequency
calculate_baselines(folder_paths)

# Now, max_time and max_frequency contain the baseline values
target_shape = (int(max_frequency), int(max_time))

# Process audio files and annotations
for folder_path in folder_paths:
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_base_name = os.path.splitext(file)[0]
            file_base_path = os.path.join(folder_path, file_base_name)
            sample_rate, samples, df = load_audio_and_annotation(file_base_path)
            segments = extract_audio_segments(samples, sample_rate, df)

            # Generate no_call segments from silent gaps
            no_call_segments = create_no_call_segments(samples, sample_rate, df)
            
            # Combine call and no-call segments
            for audio_segment in no_call_segments:
                segments.append({
                    'Selection': 'no_call',  # Use 'no_call' as the selection for silent segments
                    'Annotation': 'no_call',
                    'Begin Time (s)': 0,
                    'End Time (s)': len(audio_segment) / sample_rate,
                    'audio_segment': audio_segment,
                    'Low Freq (Hz)': 0,
                    'High Freq (Hz)': sample_rate // 2
                })
            
            # Categorize segments and add to the dataset
            for segment in segments:
                category = get_category_name(segment['Annotation'])
                segment['Category'] = category
                dataset.append(pd.DataFrame([segment]))

# Concatenate all segment data into a single DataFrame
final_df = pd.concat(dataset, ignore_index=True)

# Save spectrograms for all segments
for _, segment_info in final_df.iterrows():
    audio_segment = segment_info['audio_segment']
    frequencies, times, spectrogram_data = generate_spectrogram(
        audio_segment, sample_rate, fmin=segment_info['Low Freq (Hz)'], fmax=segment_info['High Freq (Hz)']
    )
    
    # Pad or crop the spectrogram to match the maximum size
    try:
        padded_spectrogram = pad_or_crop_spectrogram(spectrogram_data, target_shape)
    except ValueError as e:
        print(f"Error while processing {segment_info['Selection']} ({segment_info['Category']}): {e}")
        continue

    # Save spectrogram to an NPZ file
    npz_filename = f"{segment_info['Category']}_{segment_info['Begin Time (s)']}_{segment_info['End Time (s)']}.npz"
    npz_file_path = os.path.join(output_dir, npz_filename)
    np.savez(npz_file_path, spectrogram_data=padded_spectrogram, frequencies=frequencies, times=times)


Unknown annotation found: nan
Unknown annotation found: nan
Unknown annotation found: Type 4 B
Unknown annotation found: unidentified
Unknown annotation found: Trrot
Unknown annotation found: Unidentified
Unknown annotation found: Trrot
Unknown annotation found: Trrot
Unknown annotation found: Trrot
Unknown annotation found: ??
Unknown annotation found: ?
Unknown annotation found: ?
Unknown annotation found: ??
Unknown annotation found: HS Groan
Unknown annotation found: HS Groan


In [6]:
def load_dataset(output_dir, final_df):
    """
    Loads spectrogram data and their corresponding labels from a directory containing .npz files,
    linking them to the information in final_df created in code 1.

    Parameters:
        - output_dir (str): Directory containing the .npz files.
        - final_df (pd.DataFrame): The DataFrame with category and annotation info.

    Returns:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    spectrograms = []
    labels = []

    # List unique categories present in final_df for reference
    available_categories = final_df["Category"].unique()

    # Loop through each .npz file in the folder
    for file in os.listdir(output_dir):
        if file.endswith(".npz"):
            try:
                # Load the .npz file
                data = np.load(os.path.join(output_dir, file))
                spectrogram_data = data["spectrogram_data"]
                
                # Extract the annotation (category) from the filename
                file_name_parts = file.split('_')  # Split by underscore

                # Extracting category and adjusting based on filename pattern
                category_label = '_'.join(file_name_parts[:2])  # First two parts should generally match category
                
                print(f"Processing file: {file} with category: {category_label}")  # Debugging output

                # Find the corresponding row in final_df using the parsed label
                segment_row = final_df[final_df['Category'] == category_label]
                
                if not segment_row.empty:
                    category = segment_row.iloc[0]['Category']
                else:
                    # Adjust category extraction if needed
                    # If the full category like 'type_4' is not found, try to match differently
                    possible_category_label = file_name_parts[0]  # Try using just the first part, e.g., 'type'
                    segment_row = final_df[final_df['Category'] == possible_category_label]
                    
                    if not segment_row.empty:
                        category = segment_row.iloc[0]['Category']
                    else:
                        category = 'unknown'

                # Skip files with 'unknown' categories
                if category != 'unknown':
                    spectrograms.append(spectrogram_data)
                    labels.append(category)

            except (KeyError, Exception) as e:
                print(f"Error processing file {file}: {e}")
                continue

    spectrograms = np.array(spectrograms, dtype=object)  # Use dtype=object for variable-length data
    labels = np.array(labels)

    return spectrograms, labels


def summarize_data(spectrograms, labels):
    """
    Summarizes the loaded spectrograms and their corresponding labels.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    labels_df = pd.DataFrame(labels, columns=["Category"])

    num_samples = len(spectrograms)
    unique_labels = labels_df["Category"].nunique()
    category_counts = labels_df["Category"].value_counts()

    print(f"Total number of samples: {num_samples}")
    print(f"Unique categories: {unique_labels}")
    print("\nCategory distribution:")
    print(category_counts)


# Specify the directory containing your .npz files
output_dir = "C:/Users/Admin/Downloads/Machine-Learning/Project_part2/spectrogram_output"  

# Example usage with final_df
spectrograms, labels = load_dataset(output_dir, final_df)

# Print summary of the dataset
summarize_data(spectrograms, labels)


Processing file: growl_B_1140.463126349_1140.799273092.npz with category: growl_B
Processing file: growl_B_986.811773715_987.176303412.npz with category: growl_B
Processing file: growl_B_998.455311542_998.801534564.npz with category: growl_B
Processing file: g_rupe_1002.311745068_1002.850557854.npz with category: g_rupe
Processing file: g_rupe_1003.316816868_1003.837364813.npz with category: g_rupe
Processing file: g_rupe_1005.805401343_1006.312250658.npz with category: g_rupe
Processing file: g_rupe_1005.897064365_1005.997520986.npz with category: g_rupe
Processing file: g_rupe_1006.400603746_1006.893754431.npz with category: g_rupe
Processing file: g_rupe_1007.804994988_1008.325542933.npz with category: g_rupe
Processing file: g_rupe_1008.254485025_1008.875489591.npz with category: g_rupe
Processing file: g_rupe_1014.734431747_1015.369134944.npz with category: g_rupe
Processing file: g_rupe_1017.421083832_1017.873138627.npz with category: g_rupe
Processing file: g_rupe_1018.428703444

In [4]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

def preprocess_data(spectrograms, labels):
    """
    Preprocess the spectrogram data by normalizing and splitting it into training and testing sets.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.

    Returns:
        - X_train, X_test (np.ndarray): Normalized training and testing spectrogram data.
        - y_train, y_test (np.ndarray): Corresponding labels for training and testing data.
        - label_encoder (LabelEncoder): Encoder object for mapping labels to integers.
    """
    # Flatten each spectrogram if necessary for your model
    X = np.array([spec.flatten() for spec in spectrograms], dtype=np.float32)

    # Encode labels as integers
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    # Normalize spectrogram data
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_normalized = scaler.fit_transform(X)

    # Split into training and testing datasets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test, label_encoder


# Example usage
X_train, X_test, y_train, y_test, label_encoder = preprocess_data(spectrograms, labels)

# Output the results
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")
print("Classes:", label_encoder.classes_)

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import os

def load_dataset(output_dir, final_df):
    """
    Loads spectrogram data and their corresponding labels from a directory containing .npz files,
    linking them to the information in final_df created in code 1.

    Parameters:
        - output_dir (str): Directory containing the .npz files.
        - final_df (pd.DataFrame): The DataFrame with category and annotation info.

    Returns:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    spectrograms = []
    labels = []

    # List unique categories present in final_df for reference
    available_categories = final_df["Category"].unique()

    # Loop through each .npz file in the folder
    for file in os.listdir(output_dir):
        if file.endswith(".npz"):
            try:
                # Load the .npz file
                data = np.load(os.path.join(output_dir, file))
                spectrogram_data = data["spectrogram_data"]
                
                # Extract the annotation (full category) from the filename
                file_name_parts = file.split('_')  # Split by underscore
                
                # Identify the category using parts of the filename (e.g., g_rupe or rupe_A)
                category_label = '_'.join(file_name_parts[:2])  # Grab the first two parts (e.g., g_rupe or rupe_A)

                # Find the corresponding row in final_df using the parsed label
                segment_row = final_df[final_df['Category'] == category_label]
                
                if not segment_row.empty:
                    category = segment_row.iloc[0]['Category']
                else:
                    category = 'unknown'
                
                # Skip files with 'unknown' categories
                if category != 'unknown':
                    spectrograms.append(spectrogram_data)
                    labels.append(category)

            except (KeyError, Exception):
                continue

    spectrograms = np.array(spectrograms, dtype=object)  # Use dtype=object for variable-length data
    labels = np.array(labels)

    return spectrograms, labels


def preprocess_data(spectrograms, labels):
    """
    Preprocess the spectrogram data by normalizing and splitting it into training and testing sets.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.

    Returns:
        - X_train, X_test (np.ndarray): Normalized training and testing spectrogram data.
        - y_train, y_test (np.ndarray): Corresponding labels for training and testing data.
        - label_encoder (LabelEncoder): Encoder object for mapping labels to integers.
    """
    # Reshape spectrograms to include a channel dimension (height, width, 1)
    X = np.array([spec.reshape((spec.shape[0], spec.shape[1], 1)) for spec in spectrograms], dtype=np.float32)

    # Encode labels as integers
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    # Normalize spectrogram data
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_normalized = np.array([scaler.fit_transform(spec.reshape(-1, spec.shape[1])).reshape(spec.shape) for spec in X])

    # Split into training and testing datasets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test, label_encoder


def summarize_data(spectrograms, labels):
    """
    Summarizes the loaded spectrograms and their corresponding labels.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    labels_df = pd.DataFrame(labels, columns=["Category"])

    num_samples = len(spectrograms)
    unique_labels = labels_df["Category"].nunique()
    category_counts = labels_df["Category"].value_counts()

    print(f"Total number of samples: {num_samples}")
    print(f"Unique categories: {unique_labels}")
    print("\nCategory distribution:")
    print(category_counts)


def build_cnn_model(input_shape, num_classes):
    """
    Builds a CNN model for multi-class classification.

    Parameters:
        - input_shape (tuple): Shape of the input data (height, width, channels).
        - num_classes (int): Number of output classes.

    Returns:
        - model: A compiled Keras model.
    """
    model = Sequential([
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')  # Output layer
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# Example usage
output_dir = "C:/Users/Admin/Downloads/Machine-Learning/Project_part2/spectrogram_output"
spectrograms, labels = load_dataset(output_dir, final_df)

# Summarize data
summarize_data(spectrograms, labels)

# Preprocess data
X_train, X_test, y_train, y_test, label_encoder = preprocess_data(spectrograms, labels)

# Print summary of shapes and classes
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")
print("Classes:", label_encoder.classes_)

# Reshape the data for CNN (already handled during preprocessing)
input_shape = X_train.shape[1:]  # (height, width, channels)
num_classes = len(label_encoder.classes_)

# Build and compile the model
model = build_cnn_model(input_shape, num_classes)

# Summary of the model
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,  # Adjust as needed
    batch_size=32,
    verbose=1
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Save the trained model
model.save("spectrogram_cnn_model.h5")

def preprocess_data(spectrograms, labels):
    """
    Preprocesses spectrogram data for machine learning tasks. This includes filtering invalid labels, 
    normalizing the spectrogram data, and splitting it into training and testing datasets.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data, where each item is a 2D array representing a spectrogram.
        - labels (np.ndarray): Array of labels corresponding to the spectrograms.

    Returns:
        - X_train, X_test (np.ndarray): Normalized spectrogram data split into training and testing sets.
        - y_train, y_test (np.ndarray): Labels for the training and testing sets, encoded as integers.
        - label_encoder (LabelEncoder): A fitted LabelEncoder instance, useful for mapping between encoded labels and original label strings.
    """
    # Step 1: Filter out spectrograms with the "unknown" category
    valid_indices = [i for i, label in enumerate(labels) if label != "unknown"]
    filtered_spectrograms = spectrograms[valid_indices]
    filtered_labels = labels[valid_indices]

    # Step 2: Flatten spectrograms for input to machine learning models
    # Each spectrogram (2D array) is converted to a 1D array
    X = np.array([spec.flatten() for spec in filtered_spectrograms], dtype=np.float32)

    # Step 3: Encode category labels into integers
    # Converts textual category labels (e.g., "rupe_A") into integer values (e.g., 0, 1, 2, etc.)
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(filtered_labels)

    # Step 4: Normalize spectrogram data
    # Scales the spectrogram values to the range [0, 1] for better model performance
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_normalized = scaler.fit_transform(X)

    # Step 5: Split the data into training and testing sets
    # Allocates 80% of the data to training and 20% to testing
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test, label_encoder


X_train, X_test, y_train, y_test, label_encoder = preprocess_data(spectrograms, labels)

# Output the results for reference
print(f"Training data shape: {X_train.shape}")  # Shape of normalized spectrograms for training
print(f"Test data shape: {X_test.shape}")      # Shape of normalized spectrograms for testing
print(f"Training labels shape: {y_train.shape}")  # Number of labels in the training set
print(f"Test labels shape: {y_test.shape}")       # Number of labels in the testing set
print("Classes:", label_encoder.classes_)       # List of unique class labels encoded

# Define the CNN architecture
model = Sequential()

# Convolutional layer 1
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))

# Convolutional layer 2
model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# Convolutional layer 3
model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# Flatten layer
model.add(Flatten())

# Fully connected layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output layer
model.add(Dense(8, activation='softmax'))  # 8 neurons for 8 output classes

# Compile the model
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()

model = Sequential()

# First convolutional layer
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))

# Second convolutional layer
model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# Third convolutional layer
model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# Global average pooling layer to reduce the dimensions
model.add(GlobalAveragePooling1D())

# Fully connected layer
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer (8 classes)
model.add(Dense(8, activation='softmax'))

# Model summary
model.summary()

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

history = model.fit(X_train, y_train, 
                    epochs=30, 
                    batch_size=64, 
                    validation_data=(X_test, y_test))

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")




Training data shape: (2123, 14876)
Test data shape: (531, 14876)
Training labels shape: (2123,)
Test labels shape: (531,)
Classes: ['g_rupe' 'growl_B' 'no_call' 'rupe_A' 'rupe_B' 'rupe_C']


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import os

def load_dataset(output_dir, final_df):
    """
    Loads spectrogram data and their corresponding labels from a directory containing .npz files,
    linking them to the information in final_df created in code 1.

    Parameters:
        - output_dir (str): Directory containing the .npz files.
        - final_df (pd.DataFrame): The DataFrame with category and annotation info.

    Returns:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    spectrograms = []
    labels = []

    # List unique categories present in final_df for reference
    available_categories = final_df["Category"].unique()

    # Loop through each .npz file in the folder
    for file in os.listdir(output_dir):
        if file.endswith(".npz"):
            try:
                # Load the .npz file
                data = np.load(os.path.join(output_dir, file))
                spectrogram_data = data["spectrogram_data"]
                
                # Extract the annotation (full category) from the filename
                file_name_parts = file.split('_')  # Split by underscore
                
                # Identify the category using parts of the filename (e.g., g_rupe or rupe_A)
                category_label = '_'.join(file_name_parts[:2])  # Grab the first two parts (e.g., g_rupe or rupe_A)

                # Find the corresponding row in final_df using the parsed label
                segment_row = final_df[final_df['Category'] == category_label]
                
                if not segment_row.empty:
                    category = segment_row.iloc[0]['Category']
                else:
                    category = 'unknown'
                
                # Skip files with 'unknown' categories
                if category != 'unknown':
                    spectrograms.append(spectrogram_data)
                    labels.append(category)

            except (KeyError, Exception):
                continue

    spectrograms = np.array(spectrograms, dtype=object)  # Use dtype=object for variable-length data
    labels = np.array(labels)

    return spectrograms, labels


def preprocess_data(spectrograms, labels):
    """
    Preprocess the spectrogram data by normalizing and splitting it into training and testing sets.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.

    Returns:
        - X_train, X_test (np.ndarray): Normalized training and testing spectrogram data.
        - y_train, y_test (np.ndarray): Corresponding labels for training and testing data.
        - label_encoder (LabelEncoder): Encoder object for mapping labels to integers.
    """
    # Reshape spectrograms to include a channel dimension (height, width, 1)
    X = np.array([spec.reshape((spec.shape[0], spec.shape[1], 1)) for spec in spectrograms], dtype=np.float32)

    # Encode labels as integers
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    # Normalize spectrogram data
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_normalized = np.array([scaler.fit_transform(spec.reshape(-1, spec.shape[1])).reshape(spec.shape) for spec in X])

    # Split into training and testing datasets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test, label_encoder


def summarize_data(spectrograms, labels):
    """
    Summarizes the loaded spectrograms and their corresponding labels.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    labels_df = pd.DataFrame(labels, columns=["Category"])

    num_samples = len(spectrograms)
    unique_labels = labels_df["Category"].nunique()
    category_counts = labels_df["Category"].value_counts()

    print(f"Total number of samples: {num_samples}")
    print(f"Unique categories: {unique_labels}")
    print("\nCategory distribution:")
    print(category_counts)


def build_cnn_model(input_shape, num_classes):
    """
    Builds a CNN model for multi-class classification.

    Parameters:
        - input_shape (tuple): Shape of the input data (height, width, channels).
        - num_classes (int): Number of output classes.

    Returns:
        - model: A compiled Keras model.
    """
    model = Sequential([
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')  # Output layer
    ])

    # Compile the model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# Example usage
output_dir = "C:/Users/Admin/Downloads/Machine-Learning/Project_part2/spectrogram_output"
spectrograms, labels = load_dataset(output_dir, final_df)

# Summarize data
summarize_data(spectrograms, labels)

# Preprocess data
X_train, X_test, y_train, y_test, label_encoder = preprocess_data(spectrograms, labels)

# Print summary of shapes and classes
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")
print("Classes:", label_encoder.classes_)

# Reshape the data for CNN (already handled during preprocessing)
input_shape = X_train.shape[1:]  # (height, width, channels)
num_classes = len(label_encoder.classes_)

# Build and compile the model
model = build_cnn_model(input_shape, num_classes)

# Summary of the model
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,  # Adjust as needed
    batch_size=32,
    verbose=1
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Save the trained model
model.save("spectrogram_cnn_model.h5")


NameError: name 'final_df' is not defined

In [ ]:
import os
import numpy as np
import pandas as pd

def load_dataset(output_dir, final_df):
    """
    Loads spectrogram data and their corresponding labels from .npz files, 
    linking them to final_df created in code 1.

    Parameters:
        - output_dir (str): Directory containing .npz files.
        - final_df (pd.DataFrame): DataFrame with category and annotation info.

    Returns:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    spectrograms = []
    labels = []

    # Filter categories with fewer than 30 samples
    valid_categories = final_df["Category"].value_counts()
    valid_categories = valid_categories[valid_categories >= 30].index
    final_df_filtered = final_df[final_df["Category"].isin(valid_categories)]

    # List unique categories in filtered final_df
    available_categories = final_df_filtered["Category"].unique()

    for file in os.listdir(output_dir):
        if file.endswith(".npz"):
            try:
                # Load data
                data = np.load(os.path.join(output_dir, file))
                spectrogram_data = data["spectrogram_data"]

                # Extract category label from filename
                category_label = '_'.join(file.split('_')[:-2])  # Remove timestamp

                # Match with filtered final_df
                segment_row = final_df_filtered[final_df_filtered['Category'] == category_label]
                if not segment_row.empty:
                    category = segment_row.iloc[0]['Category']
                    spectrograms.append(spectrogram_data)
                    labels.append(category)

            except KeyError:
                continue

    return np.array(spectrograms, dtype=object), np.array(labels)


def summarize_data(spectrograms, labels):
    """
    Summarizes the loaded spectrograms and their corresponding labels.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    labels_df = pd.DataFrame(labels, columns=["Category"])

    num_samples = len(spectrograms)
    unique_labels = labels_df["Category"].nunique()
    category_counts = labels_df["Category"].value_counts()

    summary = (
        f"Total number of samples: {num_samples}\n"
        f"Unique categories: {unique_labels}\n\n"
        f"Category distribution:\n{category_counts.to_string()}"
    )
    print(summary)

# Example usage of the functions
spectrograms, labels = load_dataset(output_dir, final_df)
summarize_data(spectrograms, labels)

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

def preprocess_data(spectrograms, labels):
    """
    Preprocess the spectrogram data by normalizing and splitting it into training and testing sets.
    
    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.

    Returns:
        - X_train, X_test (np.ndarray): Normalized training and testing spectrogram data.
        - y_train, y_test (np.ndarray): Corresponding labels for training and testing data.
        - label_encoder (LabelEncoder): The fitted label encoder used to encode the labels.
    """
    # Flatten spectrogram data if needed (for deep learning models, you often need to reshape to 2D or 1D)
    X = np.array([spec.flatten() for spec in spectrograms], dtype=np.float32)
    
    # Encode labels as integers
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    # Normalize the spectrogram data
    scaler = MinMaxScaler(feature_range=(0, 1))  # Normalize between 0 and 1
    X_normalized = scaler.fit_transform(X)

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    # Return the necessary data (adding label_encoder to the return values)
    return X_train, X_test, y_train, y_test, label_encoder


# Example usage
X_train, X_test, y_train, y_test, label_encoder = preprocess_data(spectrograms, labels)

# Output the shapes of the data after preprocessing
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

# Reshape X_train and X_test for input into Conv1D layers
X_train_reshaped = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_reshaped = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Define the CNN architecture with a custom name
model = Sequential(name="model_1")

# 1st Convolutional Layer (modifying filter count)
model.add(Conv1D(filters=128, kernel_size=3, activation='relu', input_shape=(X_train_reshaped.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))

# 2nd Convolutional Layer (modifying filter count)
model.add(Conv1D(filters=256, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# 3rd Convolutional Layer (modifying filter count)
model.add(Conv1D(filters=512, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# Flatten the data for input into Dense layers
model.add(Flatten())

# Fully connected (Dense) Layer (modifying neurons)
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.4))  # Adjusted dropout rate for regularization

# Output Layer (for multi-class classification)
num_classes = len(np.unique(y_train))  # Set to the number of classes in your dataset
model.add(Dense(num_classes, activation='softmax'))

# Display the model summary
model.summary()
